# Training YOLOX-Nano **OBB** (Google Colab)

Trains an **oriented bounding box** YOLOX detector on the traffic-signal data (`split_obb_dataset`, 4 classes: `railroad-crossing`, `lights-on`, `lights-off`, `trefolo`). Labels are 9-column YOLO-OBB: `cls x1 y1 x2 y2 x3 y3 x4 y4` (normalized corner points).

Official Megvii YOLOX has **no** OBB head, so this uses the community fork [**buzhidaoshenme/YOLOX-OBB**](https://github.com/buzhidaoshenme/YOLOX-OBB) (YOLOX + KLD rotated loss). Its data pipeline is VOC-style XML where each `<object>` stores a rotated box as `xmin/ymin/xmax/ymax/angle` (the `cv2.minAreaRect` representation). This notebook:
1. Sets up the fork + its native deps (swig/polyiou, optional apex).
2. Converts your YOLO-OBB labels → the fork's VOC-OBB XML (via `minAreaRect`).
3. Scaffolds a **nano**-scaled OBB exp.
4. Trains / evaluates.

> ⚠️ This is a research fork, not a polished library. A few steps (apex, the class list, the exp paths) may need a small manual tweak — those cells are flagged **EDIT** and explained inline.

> **Runtime → Change runtime type → GPU** before running.

## 1. GPU check

In [ ]:
import torch, platform
print('Python    :', platform.python_version())
print('Torch     :', torch.__version__)
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU.'
print('GPU       :', torch.cuda.get_device_name(0))

## 2. Clone YOLOX-OBB & install dependencies

Installs the fork, builds the `polyiou` C++ extension (needed for rotated IoU/NMS and evaluation), and installs pycocotools.

**apex:** the fork's `--fp16` path imports NVIDIA apex, which is slow/fragile to build on Colab. This notebook trains **without** `--fp16` to avoid it. If you want fp16, uncomment the apex build.

In [ ]:
import os, sys, subprocess

REPO = '/content/YOLOX-OBB'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/buzhidaoshenme/YOLOX-OBB.git', REPO], check=True)

%cd {REPO}
!pip install -q -r requirements.txt
!pip install -q -v -e .
!pip install -q cython
!pip install -q 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'

# Build the polyiou swig extension (rotated polygon IoU).
!apt-get -qq install -y swig
%cd {REPO}/DOTA_devkit_YOLO
!swig -c++ -python polyiou.i
!python setup.py build_ext --inplace
%cd {REPO}

# --- Optional: apex, only needed if you train with --fp16 ---
# !git clone https://github.com/NVIDIA/apex /content/apex
# %cd /content/apex && pip install -v --no-cache-dir ./ && %cd {REPO}
print('Setup done.')

## 3. Mount Drive & unzip the OBB dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --- EDIT: point at your 9-column YOLO-OBB dataset zip on Drive ---
ZIP_PATH   = '/content/drive/MyDrive/split_obb_dataset/split_obb_dataset.zip'
LOCAL_ROOT = '/content/dataset'
# -----------------------------------------------------------------

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'{ZIP_PATH} not found on Drive.')
os.makedirs(LOCAL_ROOT, exist_ok=True)
!unzip -qo "{ZIP_PATH}" -d "{LOCAL_ROOT}"

DATASET_ROOT = os.path.join(LOCAL_ROOT, 'split_obb_dataset')
print('Dataset root:', DATASET_ROOT)
!ls -R "{DATASET_ROOT}" | head -20

In [ ]:
import yaml
with open(os.path.join(DATASET_ROOT, 'data.yaml')) as f:
    data_cfg = yaml.safe_load(f)
CLASSES = tuple(data_cfg['names'])
NUM_CLASSES = len(CLASSES)
print(NUM_CLASSES, 'classes:', CLASSES)

## 4. Convert YOLO-OBB → VOC-OBB XML

For each box we take the 4 normalized corner points, scale to pixels, fit a `cv2.minAreaRect`, and write the fork's `<bndbox>` schema: `xmin/ymin/xmax/ymax/angle` (axis-aligned box of size w×h centred at the rect centre, plus the rotation angle in degrees). This is exactly what the fork's `DOTA2VOC_obb.py` emits, so the VOC-OBB loader reads it natively.

Builds the VOC layout the fork expects:
```
VOCdevkit/VOC2012/
  Annotations/        <- OBB XML
  ImageSets/Main/     <- train.txt, val.txt
  JPEGImages/         <- all images
```

In [ ]:
import glob, shutil, cv2, numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom
from PIL import Image

VOC_ROOT = '/content/VOCdevkit/VOC2012'
ANN_DIR  = os.path.join(VOC_ROOT, 'Annotations')
IMG_DIR  = os.path.join(VOC_ROOT, 'JPEGImages')
SET_DIR  = os.path.join(VOC_ROOT, 'ImageSets', 'Main')
for d in (ANN_DIR, IMG_DIR, SET_DIR):
    os.makedirs(d, exist_ok=True)
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')


def _el(parent, tag, text=None):
    e = ET.SubElement(parent, tag)
    if text is not None:
        e.text = str(text)
    return e


def write_voc_obb_xml(fname, W, H, objs, out_path):
    """objs: list of (classname, xmin, ymin, xmax, ymax, angle_deg)."""
    ann = ET.Element('annotation')
    _el(ann, 'folder', 'VOC2012')
    _el(ann, 'filename', fname)
    size = _el(ann, 'size')
    _el(size, 'width', W); _el(size, 'height', H); _el(size, 'depth', 3)
    _el(ann, 'segmented', 0)
    for name, xmin, ymin, xmax, ymax, angle in objs:
        ob = _el(ann, 'object')
        _el(ob, 'name', name)
        _el(ob, 'pose', 'Unspecified')
        _el(ob, 'truncated', 0)
        _el(ob, 'difficult', 0)
        bb = _el(ob, 'bndbox')
        _el(bb, 'xmin', int(round(xmin))); _el(bb, 'ymin', int(round(ymin)))
        _el(bb, 'xmax', int(round(xmax))); _el(bb, 'ymax', int(round(ymax)))
        _el(bb, 'angle', round(float(angle), 4))
    xml = minidom.parseString(ET.tostring(ann)).toprettyxml(indent='  ')
    with open(out_path, 'w') as f:
        f.write(xml)


def obb_to_bndbox(points_px):
    """4 corner points (px) -> (xmin, ymin, xmax, ymax, angle_deg) via minAreaRect.
    Box is the unrotated w×h rect centred at (cx,cy); angle is its rotation."""
    rect = cv2.minAreaRect(np.asarray(points_px, dtype=np.float32))
    (cx, cy), (w, h), angle = rect
    return cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2, angle


def convert_split(split, dataset_root, classes):
    img_dir = os.path.join(dataset_root, split, 'images')
    lbl_dir = os.path.join(dataset_root, split, 'labels')
    img_paths = sorted(p for p in glob.glob(os.path.join(img_dir, '*'))
                       if p.lower().endswith(IMG_EXTS))
    ids = []
    for img_path in img_paths:
        fname = os.path.basename(img_path)
        stem = os.path.splitext(fname)[0]
        with Image.open(img_path) as im:
            W, H = im.size
        # copy image into the shared JPEGImages dir
        shutil.copy(img_path, os.path.join(IMG_DIR, fname))

        objs = []
        lbl_path = os.path.join(lbl_dir, stem + '.txt')
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    t = line.split()
                    if len(t) < 9:
                        continue
                    cls_id = int(float(t[0]))
                    if not (0 <= cls_id < len(classes)):
                        continue
                    coords = [float(v) for v in t[1:9]]
                    pts = [(coords[i] * W, coords[i + 1] * H) for i in range(0, 8, 2)]
                    xmin, ymin, xmax, ymax, ang = obb_to_bndbox(pts)
                    if (xmax - xmin) < 2 or (ymax - ymin) < 2:
                        continue
                    objs.append((classes[cls_id], xmin, ymin, xmax, ymax, ang))
        write_voc_obb_xml(fname, W, H, objs, os.path.join(ANN_DIR, stem + '.xml'))
        ids.append(stem)

    with open(os.path.join(SET_DIR, f'{split}.txt'), 'w') as f:
        f.write('\n'.join(ids) + '\n')
    print(f'{split}: {len(ids)} images -> XML + ImageSets/Main/{split}.txt')
    return ids


convert_split('train', DATASET_ROOT, CLASSES)
convert_split('val',   DATASET_ROOT, CLASSES)
print('VOC root:', VOC_ROOT)

In [ ]:
# Sanity check: draw the rotated boxes parsed back from one XML.
import random
from IPython.display import display

xmls = glob.glob(os.path.join(ANN_DIR, '*.xml'))
tree = ET.parse(random.choice(xmls)); root = tree.getroot()
fname = root.find('filename').text
img = cv2.imread(os.path.join(IMG_DIR, fname))
for ob in root.findall('object'):
    bb = ob.find('bndbox')
    xmin, ymin, xmax, ymax = (float(bb.find(k).text) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
    angle = float(bb.find('angle').text)
    cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
    w, h = xmax - xmin, ymax - ymin
    box = cv2.boxPoints(((cx, cy), (w, h), angle)).astype(int)
    cv2.polylines(img, [box], True, (0, 0, 255), 2)
    cv2.putText(img, ob.find('name').text, tuple(box[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
print(fname, '-', len(root.findall('object')), 'boxes')
display(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))

## 5. Register class names (**EDIT**)

The fork hardcodes class names in `yolox/data/datasets/voc_classes.py` as `VOC_CLASSES`. We overwrite it with our 4 classes (order must match the dataset).

In [ ]:
voc_classes_py = 'VOC_CLASSES = (\n' + ''.join(f'    {c!r},\n' for c in CLASSES) + ')\n'
with open(os.path.join(REPO, 'yolox/data/datasets/voc_classes.py'), 'w') as f:
    f.write(voc_classes_py)
print(voc_classes_py)

## 6. Nano-scaled OBB exp (**EDIT**)

The fork ships an `_s` OBB config at `exps/example/yolox_voc/yolox_dota_s_obb_kld.py`. We copy it to a nano variant and patch: model size → nano (`depth=0.33, width=0.25`), `num_classes`, and the VOC `data_dir`. All the OBB/KLD loss + VOC-OBB dataloader wiring is inherited unchanged.

We print the result — **check it**: confirm `self.data_dir` points at `/content/VOCdevkit` (or the `VOC2012` parent), and that `image_sets` / year match what we built (`2012`, `train`/`val`). Adjust the patch below if the base file uses different attribute names.

In [ ]:
import re
BASE_EXP = os.path.join(REPO, 'exps/example/yolox_voc/yolox_dota_s_obb_kld.py')
NANO_EXP = os.path.join(REPO, 'exps/example/yolox_voc/yolox_nano_obb_kld.py')

src = open(BASE_EXP).read()
print('--- BASE EXP (for reference) ---\n', src)

# Patch model size, class count, and dataset root.
src = re.sub(r'self\.depth\s*=.*', 'self.depth = 0.33', src)
src = re.sub(r'self\.width\s*=.*', 'self.width = 0.25', src)
if 'self.depthwise' in src:
    src = re.sub(r'self\.depthwise\s*=.*', 'self.depthwise = True', src)
else:
    src = src.replace('self.width = 0.25', 'self.width = 0.25\n        self.depthwise = True')
src = re.sub(r'self\.num_classes\s*=.*', f'self.num_classes = {NUM_CLASSES}', src)
# Point the VOC data_dir at our converted dataset (VOCdevkit parent).
src = re.sub(r'(data_dir\s*=\s*).*', r"\1'/content/VOCdevkit'", src)

with open(NANO_EXP, 'w') as f:
    f.write(src)
print('\n--- WROTE NANO EXP ---\n', src)

## 7. Train

From-scratch (no COCO-pretrained nano weights ship with the fork). To warm-start, pass `-c <yolox weights>.pth`. Trained without `--fp16` to skip apex; add it (and build apex above) if you want mixed precision.

In [ ]:
%cd {REPO}
!python tools/train.py -f exps/example/yolox_voc/yolox_nano_obb_kld.py -d 1 -b 16

## 8. Evaluate (rotated mAP)

Runs the fork's eval, then merges results and scores with the DOTA devkit.

In [ ]:
CKPT = 'YOLOX_outputs/yolox_nano_obb_kld/latest_ckpt.pth'
%cd {REPO}
!python tools/eval.py -f exps/example/yolox_voc/yolox_nano_obb_kld.py -d 1 -b 16 -c {CKPT}
# Then (per the fork README) merge + score:
# !python DOTA_devkit_YOLO/ResultMerge.py
# !python DOTA_devkit_YOLO/dota_v1.5_evaluation_task1.py

## 9. Save checkpoint to Drive

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/yolox_obb_results'
os.makedirs(SAVE_DIR, exist_ok=True)
src_ckpt = os.path.join(REPO, CKPT)
if os.path.exists(src_ckpt):
    shutil.copy(src_ckpt, os.path.join(SAVE_DIR, 'yolox_nano_obb_best.pth'))
    print('Saved to', SAVE_DIR)
else:
    print('Checkpoint not found yet:', src_ckpt)

---
### Notes & gotchas
- **Why a fork:** official YOLOX has no rotated head. [buzhidaoshenme/YOLOX-OBB](https://github.com/buzhidaoshenme/YOLOX-OBB) adds an OBB head with KLD loss and a VOC-XML (`xmin/ymin/xmax/ymax/angle`) pipeline.
- **Angle convention:** boxes are stored as the `cv2.minAreaRect` of your 4 corner points. The fork decodes the same way, so the round-trip is consistent (the sanity-check cell verifies it visually).
- **If training errors on the dataloader:** open the printed nano exp and confirm `data_dir`, the VOC `year` (we used `2012`), and the train/val `image_sets` names match `ImageSets/Main/{train,val}.txt`. The base `_s` exp is the source of truth for attribute names.
- **apex / fp16:** left off by default. Build apex (cell 2) and add `--fp16` only if you need it.
- **Nano vs S:** the fork has no official nano-OBB config; we scale `depth/width` down and enable depthwise. If accuracy is poor on small rotated signs, try the `_s` size (`depth=0.33, width=0.50`).
- **Cleaner alternative:** your 9-column labels are native [Ultralytics YOLO-OBB](https://docs.ultralytics.com/tasks/obb) format — `YOLO('yolo11n-obb.pt').train(data=...)` trains in a few lines if you don't strictly need the YOLOX architecture.